In [1]:
import numpy as np
import pandas as pd
from utils import string_to_xml_file
import xml.etree.ElementTree as ET
from pathlib import Path
from xml.dom import minidom

In [2]:
proj_path = Path("/home/hyuntae-choi/gcam-core")
xml_path = proj_path / "input" / "gcamdata" / "xml"
db_path = proj_path / "output"

# Korea's Regulation to Promote Zero Energy Buildings

## Current Policy

A ZEB Regulation adopted for new buildings. Public buildings should achieve energy self-sufficiency rate(ESSR) at least 20% for 2023, 40% for 2025, 60% for 2030. For year 2020, this regulation was applied only for buildings with floorspace more than 500m^2. Scenario adopts ZEB from year 2025 and we assume that after 2030, 60% requirement is applied. Private buildings with floorspace more than 1,000 m^2 should achieve 20% of ESSR in 2025. After 2030 the requirement is applied to ones with floor space more than 500 m^2.

We model ZEB regulation by two steps: 1. calculate the total floor space required to attain full energy self-sufficiency. For example if a building with floor space 1000 m^2 attains 30% of ESSR, then we regard a building with floor space 300 m^2 attains full sufficiency. 2. We use unit energy consumption EJ/m^2 to calculate how much energy is needed. In GCAM, we assume energy is supplied by rooftop PV technology.

We refer two external dataset `/ext/240417(별첨)_23년_기준_건축_인허가_통계(건축정책과).xlsx` and `ext/240417(별첨)_23년_기준_건축물_현황_통계(건축정책과).xlsx` for estimation.

### Calculate ZEB share

In [3]:
tot_pr_floor_space = 4227660684.322 - 376645408.432
new_pr_floor_space = 147394196.42337 - 9195076.7802
new_pr_ratio = new_pr_floor_space / tot_pr_floor_space
new_pr_ratio

0.035886411697297435

In [4]:
fs_gt_1000_share = 0.35
fs_gt_500_share = 0.6
ss_ratio_g5 = 0.2
new_pr_ratio_ss_25 = fs_gt_1000_share * new_pr_ratio * ss_ratio_g5 * 5
new_pr_ratio_ss_30 = new_pr_ratio_ss_25 + (new_pr_ratio * fs_gt_500_share) * ss_ratio_g5 * 5
new_pr_ratio_ss_35 = new_pr_ratio_ss_30 + (new_pr_ratio * fs_gt_500_share) * ss_ratio_g5 * 5
new_pr_ratio_ss_40 = new_pr_ratio_ss_35 + (new_pr_ratio * fs_gt_500_share) * ss_ratio_g5 * 5
print(new_pr_ratio_ss_25, new_pr_ratio_ss_30, new_pr_ratio_ss_35, new_pr_ratio_ss_40)

0.012560244094054103 0.03409209111243257 0.05562393813081103 0.0771557851491895


In [5]:
ss_ratio = 0.2
private_vals = [fs_gt_1000_share * new_pr_ratio, fs_gt_500_share * ss_ratio, fs_gt_500_share * ss_ratio, fs_gt_500_share * ss_ratio]

In [6]:
tot_public_floor_space = 376645408.432
new_public_floor_space = 9195076.7802
new_public_ratio = new_public_floor_space / tot_public_floor_space
new_public_ratio

0.0244130860866716

In [7]:
ss_ratio_25 = 0.4
ss_ratio_30 = 0.6
new_public_ratio_ss_25 = fs_gt_500_share * new_public_ratio * ss_ratio_25 * 5
new_public_ratio_ss_30 = new_public_ratio_ss_25 + (fs_gt_500_share * ss_ratio_30 * new_public_ratio) * 5
new_public_ratio_ss_35 = new_public_ratio_ss_30 + (fs_gt_500_share * ss_ratio_30 * new_public_ratio) * 5
new_public_ratio_ss_40 = new_public_ratio_ss_35 + (fs_gt_500_share * ss_ratio_30 * new_public_ratio) * 5
print(new_public_ratio_ss_25, new_public_ratio_ss_30, new_public_ratio_ss_35, new_public_ratio_ss_40)

0.02929570330400592 0.0732392582600148 0.1171828132160237 0.16112636817203257


In [8]:
public_vals = [ss_ratio_25, ss_ratio_30, ss_ratio_30, ss_ratio_30]

In [9]:
pr_share = 0.911
serZebShare = pd.Series([
    new_pr_ratio_ss_25 * pr_share + new_public_ratio_ss_25 * (1-pr_share),
    new_pr_ratio_ss_30 * pr_share + new_public_ratio_ss_30 * (1-pr_share),
    new_pr_ratio_ss_35 * pr_share + new_public_ratio_ss_35 * (1-pr_share),
    new_pr_ratio_ss_40 * pr_share + new_public_ratio_ss_40 * (1-pr_share),
], index=[2025, 2030, 2035, 2040])
serZebShare

2025    0.014050
2030    0.037576
2035    0.061103
2040    0.084629
dtype: float64

In [10]:
serZebShare[2025]

np.float64(0.014049699963739814)

In [11]:
building_file_path = xml_path / "building_det.xml"
tree = ET.parse(building_file_path)
root = tree.getroot()  # Get the root element of the XML
korea = root.find(".//region[@name='South Korea']")
comm = root.find(".//gcam-consumer[@name='comm']")
global_technology_database = root.find(".//global-technology-database")

In [12]:
# Create the new root for the reproduced XML
new_root = ET.Element("scenario")
new_world = ET.SubElement(new_root, "world")
new_korea = ET.SubElement(new_world, "region", {'name': "South Korea"})
for gcam_consumer in korea.findall(".//gcam-consumer"):
    gcam_consumerNm = gcam_consumer.get("name")
    new_gcam_consumer = ET.SubElement(new_korea, 'gcam-consumer', {'name': gcam_consumerNm})

    for nodeInput in gcam_consumer.findall(".//nodeInput"):
        nodeInputNm = nodeInput.get('name')
        new_nodeInput = ET.SubElement(new_gcam_consumer, 'nodeInput', {'name': nodeInputNm})

        for building_node_input in nodeInput.findall(".//building-node-input"):
            building_node_inputNm = building_node_input.get('name')
            new_building_node_input = ET.SubElement(new_nodeInput, 'building-node-input', {'name': building_node_inputNm})

            # base_value = float(building_node_input.find(".//shell-conductance[@year='2020']").text)

            for shell_conductance in building_node_input.findall(".//shell-conductance"):
                year = int(shell_conductance.get('year'))
                if (year > 2035) or (year < 2025):
                    continue
                new_shell_conductance = ET.SubElement(new_building_node_input, 'shell-conductance', {'year': str(year)})
                base_value = float(shell_conductance.text)
                new_value = base_value * (1 - serZebShare[year])
                new_shell_conductance.text = f"{new_value:.3f}"

In [13]:
outfile_path = proj_path / "input" / "policy" / "korea-2035" / "buildings" / "zeb_cp.xml"

# save
xml_string = ET.tostring(new_root, encoding="unicode")
string_to_xml_file(xml_string, outfile_path)

XML file '/home/hyuntae-choi/gcam-core/input/policy/korea-2035/buildings/zeb_cp.xml' created successfully with proper indentation and no extra newlines.


# Enhanced Policy

Private buildings are regulated by the same condition for the public buildings after 2030

In [14]:
ss_ratio = 0.2
ss_ratio_30 = 0.4
new_pr_ratio_ss_25 = fs_gt_1000_share * new_pr_ratio * ss_ratio * 5
new_pr_ratio_ss_30 = new_pr_ratio_ss_25 + (new_pr_ratio * fs_gt_500_share) * ss_ratio_30 * 5
new_pr_ratio_ss_35 = new_pr_ratio_ss_30 + (new_pr_ratio * fs_gt_500_share) * ss_ratio_30 * 5
new_pr_ratio_ss_40 = new_pr_ratio_ss_35 + (new_pr_ratio * fs_gt_500_share) * ss_ratio_30 * 5
print(new_pr_ratio_ss_25, new_pr_ratio_ss_30, new_pr_ratio_ss_35, new_pr_ratio_ss_40)

0.012560244094054103 0.05562393813081103 0.09868763216756796 0.1417513262043249


In [15]:
pr_share = 0.911
serZebShare = pd.Series([
    new_pr_ratio_ss_25 * pr_share + new_public_ratio_ss_25 * (1-pr_share),
    new_pr_ratio_ss_30 * pr_share + new_public_ratio_ss_30 * (1-pr_share),
    new_pr_ratio_ss_35 * pr_share + new_public_ratio_ss_35 * (1-pr_share),
    new_pr_ratio_ss_40 * pr_share + new_public_ratio_ss_40 * (1-pr_share),
], index=[2025, 2030, 2035, 2040])
serZebShare

2025    0.014050
2030    0.057192
2035    0.100334
2040    0.143476
dtype: float64

In [16]:
# Create the new root for the reproduced XML
new_root = ET.Element("scenario")
new_world = ET.SubElement(new_root, "world")
new_korea = ET.SubElement(new_world, "region", {'name': "South Korea"})
for gcam_consumer in korea.findall(".//gcam-consumer"):
    gcam_consumerNm = gcam_consumer.get("name")
    new_gcam_consumer = ET.SubElement(new_korea, 'gcam-consumer', {'name': gcam_consumerNm})

    for nodeInput in gcam_consumer.findall(".//nodeInput"):
        nodeInputNm = nodeInput.get('name')
        new_nodeInput = ET.SubElement(new_gcam_consumer, 'nodeInput', {'name': nodeInputNm})

        for building_node_input in nodeInput.findall(".//building-node-input"):
            building_node_inputNm = building_node_input.get('name')
            new_building_node_input = ET.SubElement(new_nodeInput, 'building-node-input', {'name': building_node_inputNm})

            # base_value = float(building_node_input.find(".//shell-conductance[@year='2020']").text)

            for shell_conductance in building_node_input.findall(".//shell-conductance"):
                year = int(shell_conductance.get('year'))
                if (year > 2035) or (year < 2025):
                    continue
                new_shell_conductance = ET.SubElement(new_building_node_input, 'shell-conductance', {'year': str(year)})
                base_value = float(shell_conductance.text)
                new_value = base_value * (1 - serZebShare[year])
                new_shell_conductance.text = f"{new_value:.3f}"

In [17]:
outfile_path = proj_path / "input" / "policy" / "korea-2035" / "buildings" / "zeb_ep.xml"

# save
xml_string = ET.tostring(new_root, encoding="unicode")
string_to_xml_file(xml_string, outfile_path)

XML file '/home/hyuntae-choi/gcam-core/input/policy/korea-2035/buildings/zeb_ep.xml' created successfully with proper indentation and no extra newlines.
